In [3]:
# 1) Imports
# Run this cell first. If you restart the kernel, re-run from here downward.

import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI


API key found and looks good so far!


In [ ]:
# 2) Simple website scraper (same idea as week1/scraper.py)
# Uses requests + BeautifulSoup. JS-heavy / Cloudflare-protected sites often fail.

from bs4 import BeautifulSoup
import requests


# Browser-like headers so some sites accept the request
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        # Drop noisy tags before extracting text
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]


In [3]:
# 3) Load OPENAI_API_KEY from the project .env file
# Needs a key that starts with sk-proj-

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [ ]:
# 4) System prompt = role + tone + output format for the model
# Tweak this to change style (e.g. serious, Spanish, bullet-only).

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""


In [ ]:
# 5) User prompt prefix = the task instructions
# The scraped page text gets appended after this in messages_for().

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""


In [ ]:
# 6) Build the OpenAI messages list: system + user
# This is the standard chat format most providers share.

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]


## Time to bring it together - the API for OpenAI is very simple!

**Run order tip:** create the `openai` client *before* calling `summarize()`, or you will get `NameError: name 'openai' is not defined`.


In [ ]:
# 7) Create the OpenAI client + summarize() helper
# IMPORTANT: run this cell before summarize(...) / display_summary(...)
# Flow: URL -> scrape -> messages -> chat.completions -> text

openai = OpenAI()  # uses OPENAI_API_KEY from the environment


def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages_for(website),
    )
    return response.choices[0].message.content


In [8]:
# 8) Quick smoke test — returns plain text (not rendered markdown)

summarize("https://edwarddonner.com")


'# Edward site\n\nSnarky summary with leftover output...'

In [ ]:
# 9) Pretty printer — same as summarize(), but renders markdown in the notebook

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))


In [10]:
# 10) Rendered summary for Ed's site

display_summary("https://edwarddonner.com")


# Leftover output

This output was intentionally not cleared.

# Let's try more websites

Note that this will only work on websites that can be scraped using this simplistic approach.


In [ ]:
# 11) Try another site (news pages can be noisy / partially blocked)

display_summary("https://cnn.com")


In [ ]:
# 12) Another site

display_summary("https://anthropic.com")
